In [ ]:
import pandas as pd
from sklearn.metrics import precision_score, recall_score
from google.colab import files
from sklearn.linear_model import LogisticRegression
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC


In [ ]:
uploaded = files.upload()
exploratory_df = pd.read_csv(list(uploaded.keys())[0])

Saving exploratory-packages-highest-ctr.csv to exploratory-packages-highest-ctr.csv


In [ ]:
uploaded = files.upload()
confirmatory_df = pd.read_csv(list(uploaded.keys())[0])

Saving confirmatory-packages-highest-ctr.csv to confirmatory-packages-highest-ctr.csv


In [ ]:
uploaded = files.upload()
holdout_df = pd.read_csv(list(uploaded.keys())[0])

Saving holdout-packages-highest-ctr.csv to holdout-packages-highest-ctr.csv


In [ ]:
max_ctr_per_test = confirmatory_df.groupby('clickability_test_id')['ctr'].transform('max')
confirmatory_df['is_highest_ctr'] = confirmatory_df['ctr'] == max_ctr_per_test
confirmatory_df

,package_id,created_at,test_week,clickability_test_id,headline,eyecatcher_id,impressions,clicks,ctr,ctr_demeaned,first_place,winner,is_highest_ctr
0,11,2014-11-20 11:33:26.475,201446,546dd17e26714c82cc00001c,"Let’s See … Hire Cops, Pay Teachers, Buy Books...",546dce659ad54ec65b000041,3118,8,0.002566,-1.342477,False,False,False
1,12,2014-11-20 15:00:01.032,201446,546e01d626714c6c4400004e,People Sent This Lesbian Questions And Her Rai...,546d1b4bfd3617f091000041,4587,130,0.028341,-0.597779,False,False,False
2,13,2014-11-20 11:33:51.973,201446,546dd17e26714c82cc00001c,$3 Million Is What It Takes For A State To Leg...,546dce659ad54ec65b000041,3017,19,0.006298,0.265248,False,False,False
3,14,2014-11-20 11:34:12.107,201446,546dd17e26714c82cc00001c,The Fact That Sometimes Innocent People Are Ex...,546dce659ad54ec65b000041,2974,26,0.008742,1.318477,True,False,True
4,15,2014-11-20 11:34:33.935,201446,546dd17e26714c82cc00001c,Reason #351 To End The Death Penalty: It Costs...,546dce659ad54ec65b000041,3050,10,0.003279,-1.035337,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
57408,150810,2014-11-20 06:16:50.918,201446,546d86879ad54ec65b000038,Life For The People Of North Korea Is Worse Th...,545c477e74bfe7589400000b,6825,176,0.025788,0.972361,True,False,False
57409,150811,2014-11-20 06:17:39.811,201446,546d86879ad54ec65b000038,Kim Jong Un Would Really Hate For You To Watch...,545c477e74bfe7589400000b,6744,177,0.026246,1.072661,False,False,True
57410,150812,2014-11-20 06:18:48.894,201446,546d86879ad54ec65b000038,"When She Was 9 Years Old, This Girl Saw A Woma...",545c477e74bfe7589400000b,6683,154,0.023044,0.371443,False,False,False
57411,150814,2014-11-20 06:21:12.691,201446,546d86879ad54ec65b000038,This Incredible Young Woman Walked Through The...,545c477e74bfe7589400000b,6826,132,0.019338,-0.440085,False,False,False


In [ ]:
max_ctr_per_test = holdout_df.groupby('clickability_test_id')['ctr'].transform('max')
holdout_df['is_highest_ctr'] = holdout_df['ctr'] == max_ctr_per_test
holdout_df

,package_id,created_at,test_week,clickability_test_id,headline,eyecatcher_id,impressions,clicks,ctr,ctr_demeaned,first_place,winner,is_highest_ctr
0,37,2014-11-20 12:59:23.366,201446,546de5a784ad3834f000004a,The Fact That Sometimes Innocent People Are Ex...,546de0d084ad380b59000031,2996,53,0.017690,0.239907,False,False,False
1,38,2014-11-20 12:59:49.054,201446,546de5a784ad3834f000004a,The Simple Fact That Some Innocent People Are ...,546de0d084ad380b59000031,2991,35,0.011702,-1.133612,False,False,False
2,39,2014-11-20 13:00:19.893,201446,546de5a784ad3834f000004a,Sometimes Innocent People Are Executed. I Thin...,546de0d084ad380b59000031,3100,49,0.015806,-0.192162,False,False,False
3,40,2014-11-20 13:00:34.29,201446,546de5a784ad3834f000004a,The Fact That Sometimes Innocent People Are Ex...,546de0d084ad380b59000031,3010,44,0.014618,-0.464759,False,False,False
4,41,2014-11-20 13:01:35.944,201446,546de5a784ad3834f000004a,Sometimes Innocent People Are Executed. That's...,546de0d084ad380b59000031,3119,73,0.023405,1.550626,True,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
12469,150742,2014-11-20 01:18:02.777,201446,546d414a84ad380752000026,4 People Say Things No One Ever Should Have To...,546cae83badeb5294a000023,3097,25,0.008072,-0.613651,False,False,False
12470,150743,2014-11-20 01:19:03.29,201446,546d414a84ad380752000026,These 4 People Take Turns Saying Things That S...,546cae83badeb5294a000023,3022,19,0.006287,-1.318123,False,False,False
12471,150744,2014-11-20 01:20:18.132,201446,546d414a84ad380752000026,"The More And More These 4 People Talk, The Mor...",546cae83badeb5294a000023,2953,32,0.010836,0.477177,False,False,False
12472,150746,2014-11-20 01:20:51.999,201446,546d414a84ad380752000026,"The More And More These 4 People Talk, The Mor...",546cae83badeb5294a000023,3174,32,0.010082,0.179412,False,False,False


In [ ]:
# Removes tests that have the same headline as it is not useful for LLM
tests_summary = exploratory_df.groupby('clickability_test_id').agg(
    n_headlines=('headline', 'nunique'),
    n_eyecatchers=('eyecatcher_id', 'nunique')
).reset_index()
conditions = [
    (tests_summary['n_headlines'] == 1) & (tests_summary['n_eyecatchers'] == 1),
    (tests_summary['n_headlines'] == 1) & (tests_summary['n_eyecatchers'] > 1),
    (tests_summary['n_headlines'] > 1) & (tests_summary['n_eyecatchers'] == 1),
    (tests_summary['n_headlines'] > 1) & (tests_summary['n_eyecatchers'] > 1),
    (tests_summary['n_headlines'] > 1)
]
exploratory_df = exploratory_df[exploratory_df['clickability_test_id'].isin(tests_summary[conditions[4]]['clickability_test_id'])]
tests_summary = confirmatory_df.groupby('clickability_test_id').agg(
    n_headlines=('headline', 'nunique'),
    n_eyecatchers=('eyecatcher_id', 'nunique')
).reset_index()
conditions = [
    (tests_summary['n_headlines'] == 1) & (tests_summary['n_eyecatchers'] == 1),
    (tests_summary['n_headlines'] == 1) & (tests_summary['n_eyecatchers'] > 1),
    (tests_summary['n_headlines'] > 1) & (tests_summary['n_eyecatchers'] == 1),
    (tests_summary['n_headlines'] > 1) & (tests_summary['n_eyecatchers'] > 1),
    (tests_summary['n_headlines'] > 1)

]
confirmatory_df = confirmatory_df[confirmatory_df['clickability_test_id'].isin(tests_summary[conditions[4]]['clickability_test_id'])]
tests_summary = holdout_df.groupby('clickability_test_id').agg(
    n_headlines=('headline', 'nunique'),
    n_eyecatchers=('eyecatcher_id', 'nunique')
).reset_index()
conditions = [
    (tests_summary['n_headlines'] == 1) & (tests_summary['n_eyecatchers'] == 1),
    (tests_summary['n_headlines'] == 1) & (tests_summary['n_eyecatchers'] > 1),
    (tests_summary['n_headlines'] > 1) & (tests_summary['n_eyecatchers'] == 1),
    (tests_summary['n_headlines'] > 1) & (tests_summary['n_eyecatchers'] > 1),
    (tests_summary['n_headlines'] > 1)

]
holdout_df = holdout_df[holdout_df['clickability_test_id'].isin(tests_summary[conditions[4]]['clickability_test_id'])]


My original method of choice was going to be using an LLM such as GPT-OSS and to however my cpu was unable to handle it so we had to pivot to Sentence Transformers and TFID

In [ ]:
# TFID Vectorizer
tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 1),
    min_df = 5
)
X = tfidf.fit_transform(exploratory_df["headline"].fillna(""))

In [ ]:
# Logistic regression for TFID
y = exploratory_df["is_highest_ctr"]
tfidLR = LogisticRegression(max_iter=1000)
tfidLR.fit(X, y)

LogisticRegression(max_iter=1000)

In [ ]:
# Precision and Recall for TFID Logistic Regression
yConf = confirmatory_df["is_highest_ctr"]
xConf = tfidf.transform(confirmatory_df["headline"].fillna(""))
confirmatoryPrecision = precision_score(yConf, tfidLR.predict(xConf))
print("Confirmatory Precision:", confirmatoryPrecision)
confirmatoryRecall = recall_score(yConf, tfidLR.predict(xConf))
print("Confirmatory Recall:", confirmatoryRecall)
yHold = holdout_df["is_highest_ctr"]
xHold = tfidf.transform(holdout_df["headline"].fillna(""))
holdoutPrecision = precision_score(yHold, tfidLR.predict(xHold))
print("Holdout Precision:", holdoutPrecision)
holdoutRecall = recall_score(yHold, tfidLR.predict(xHold))
print("Holdout Recall:", holdoutRecall)

Confirmatory Precision: 0.2796610169491525
Confirmatory Recall: 0.002638522427440633
Holdout Precision: 0.16
Holdout Recall: 0.0014814814814814814


In [ ]:
# SVM for TFID
tfidSVM = LinearSVC()
tfidSVM.fit(X, y)

LinearSVC()

In [ ]:
# Precision and Recall for TFID Linear SVM
yConf = confirmatory_df["is_highest_ctr"]
xConf = tfidf.transform(confirmatory_df["headline"].fillna(""))
confirmatoryPrecision = precision_score(yConf, tfidSVM.predict(xConf))
print("Confirmatory Precision:", confirmatoryPrecision)
confirmatoryRecall = recall_score(yConf, tfidSVM.predict(xConf))
print("Confirmatory Recall:", confirmatoryRecall)
yHold = holdout_df["is_highest_ctr"]
xHold = tfidf.transform(holdout_df["headline"].fillna(""))
holdoutPrecision = precision_score(yHold, tfidSVM.predict(xHold))
print("Holdout Precision:", holdoutPrecision)
holdoutRecall = recall_score(yHold, tfidSVM.predict(xHold))
print("Holdout Recall:", holdoutRecall)

Confirmatory Precision: 0.2551274083281541
Confirmatory Recall: 0.06564323978572
Holdout Precision: 0.24316939890710382
Holdout Recall: 0.06592592592592593


In [ ]:
# Sentence Transformer model 1 - all-MiniLM-L6-v2
model = SentenceTransformer("all-MiniLM-L6-v2")
X = model.encode(
    exploratory_df["headline"].fillna("").tolist(),
    batch_size=64
)

In [ ]:
# Confirmatory dataset
XConf = model.encode(
    confirmatory_df["headline"].fillna("").tolist(),
    batch_size=64
)

In [ ]:
# Holdout dataset
XHold = model.encode(
    holdout_df["headline"].fillna("").tolist(),
    batch_size=64
)

In [ ]:
# Precision and Recall for first sentence transformer using Logistic Regression
ySentence = exploratory_df["is_highest_ctr"]
sentenceLR = LogisticRegression(max_iter=1000)
sentenceLR.fit(X, ySentence)
confirmatoryPrecision = precision_score(yConf, sentenceLR.predict(XConf))
print("Confirmatory Precision:", confirmatoryPrecision)
confirmatoryRecall = recall_score(yConf, sentenceLR.predict(XConf))
print("Confirmatory Recall:", confirmatoryRecall)
holdoutPrecision = precision_score(yHold, sentenceLR.predict(XHold))
print("Holdout Precision:", holdoutPrecision)
holdoutRecall = recall_score(yHold, sentenceLR.predict(XHold))
print("Holdout Recall:", holdoutRecall)

Confirmatory Precision: 0.17647058823529413
Confirmatory Recall: 0.00023986567522187575
Holdout Precision: 0.0
Holdout Recall: 0.0


In [ ]:
# Precision and Recall for first sentence transformer using SVM
sentenceSVM = LinearSVC()
sentenceSVM.fit(X, ySentence)
confirmatoryPrecision = precision_score(yConf, sentenceSVM.predict(XConf))
print("Confirmatory Precision:", confirmatoryPrecision)
confirmatoryRecall = recall_score(yConf, sentenceSVM.predict(XConf))
print("Confirmatory Recall:", confirmatoryRecall)
holdoutPrecision = precision_score(yHold, sentenceSVM.predict(XHold))
print("Holdout Precision:", holdoutPrecision)
holdoutRecall = recall_score(yHold, sentenceSVM.predict(XHold))
print("Holdout Recall:", holdoutRecall)

Confirmatory Precision: 0.25
Confirmatory Recall: 0.0003997761253697929
Holdout Precision: 0.5
Holdout Recall: 0.00037037037037037035


In [ ]:
# Sentence Transformer model 2 - sentence-t5-base
model = SentenceTransformer("sentence-t5-base")
X = model.encode(
    exploratory_df["headline"].fillna("").tolist(),
    batch_size=64
)

modules.json:   0%|          | 0.00/461 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

2_Dense/model.safetensors:   0%|          | 0.00/2.36M [00:00<?, ?B/s]

In [ ]:
# Confirmatory dataset
XConf = model.encode(
    confirmatory_df["headline"].fillna("").tolist(),
    batch_size=64
)

In [ ]:
# Holdout dataset
XHold = model.encode(
    holdout_df["headline"].fillna("").tolist(),
    batch_size=64
)

In [ ]:
# Precision and Recall for second sentence transformer using Logistic Regression
sentenceLR.fit(X, ySentence)
confirmatoryPrecision = precision_score(yConf, sentenceLR.predict(XConf))
print("Confirmatory Precision:", confirmatoryPrecision)
confirmatoryRecall = recall_score(yConf, sentenceLR.predict(XConf))
print("Confirmatory Recall:", confirmatoryRecall)
holdoutAccuracy = accuracy_score(yHold, sentenceLR.predict(XHold))
holdoutPrecision = precision_score(yHold, sentenceLR.predict(XHold))
print("Holdout Precision:", holdoutPrecision)
holdoutRecall = recall_score(yHold, sentenceLR.predict(XHold))
print("Holdout Recall:", holdoutRecall)

Confirmatory Precision: 0.3333333333333333
Confirmatory Recall: 7.995522507395858e-05
Holdout Precision: 0.0
Holdout Recall: 0.0


In [ ]:
# Precision and Recall for second sentence transformer using SVM
sentenceSVM.fit(X, ySentence)
confirmatoryPrecision = precision_score(yConf, sentenceSVM.predict(XConf))
print("Confirmatory Precision:", confirmatoryPrecision)
confirmatoryRecall = recall_score(yConf, sentenceSVM.predict(XConf))
print("Confirmatory Recall:", confirmatoryRecall)
holdoutPrecision = precision_score(yHold, sentenceSVM.predict(XHold))
print("Holdout Precision:", holdoutPrecision)
holdoutRecall = recall_score(yHold, sentenceSVM.predict(XHold))
print("Holdout Recall:", holdoutRecall)

Confirmatory Precision: 0.3157894736842105
Confirmatory Recall: 0.0004797313504437515
Holdout Precision: 0.3333333333333333
Holdout Recall: 0.0011111111111111111


As we can see the precision and recall is best when using the second sentence transformer using Linear SVM.